In [1]:
import heapq

# The Knowledge Base (KB) mapping Kenyan cities and their highway distances (in km)
knowledge_base = {
    'Nairobi': {'Naivasha': 92.8, 'Thika': 43.0, 'Narok': 142.0},
    'Thika': {'Nyeri': 70.0},
    'Nyeri': {'Nakuru': 102.0},
    'Naivasha': {'Nakuru': 70.5},
    'Narok': {'Bomet': 98.0},
    'Bomet': {'Kisii': 78.0},
    'Nakuru': {'Kericho': 107.0, 'Eldoret': 157.0},
    'Kericho': {'Kisumu': 83.9},
    'Eldoret': {'Kisumu': 120.0},
    'Kisii': {'Kisumu': 107.0},
    'Kisumu': {} # Terminal/Goal node
}

def display_kb():
    print("\n--- Current Environmental Knowledge Base (Map) ---")
    for source, destinations in knowledge_base.items():
        for dest, cost in destinations.items():
            print(f"Highway: {source:12} ➜ {dest:12} (Distance: {cost} km)")
    print("--------------------------------------------------")

display_kb()


--- Current Environmental Knowledge Base (Map) ---
Highway: Nairobi      ➜ Naivasha     (Distance: 92.8 km)
Highway: Nairobi      ➜ Thika        (Distance: 43.0 km)
Highway: Nairobi      ➜ Narok        (Distance: 142.0 km)
Highway: Thika        ➜ Nyeri        (Distance: 70.0 km)
Highway: Nyeri        ➜ Nakuru       (Distance: 102.0 km)
Highway: Naivasha     ➜ Nakuru       (Distance: 70.5 km)
Highway: Narok        ➜ Bomet        (Distance: 98.0 km)
Highway: Bomet        ➜ Kisii        (Distance: 78.0 km)
Highway: Nakuru       ➜ Kericho      (Distance: 107.0 km)
Highway: Nakuru       ➜ Eldoret      (Distance: 157.0 km)
Highway: Kericho      ➜ Kisumu       (Distance: 83.9 km)
Highway: Eldoret      ➜ Kisumu       (Distance: 120.0 km)
Highway: Kisii        ➜ Kisumu       (Distance: 107.0 km)
--------------------------------------------------


In [2]:
def add_new_route_fact(source, destination, distance):
    """Add a new highway route to the KB."""
    if source not in knowledge_base:
        knowledge_base[source] = {}
    if destination not in knowledge_base:
        knowledge_base[destination] = {}
        
    knowledge_base[source][destination] = distance
    print(f"Added Knowledge: Highway from '{source}' to '{destination}' is {distance} km.")

def block_route(source, destination):
    """Remove a route from the KB due to a roadblock or heavy traffic."""
    if source in knowledge_base and destination in knowledge_base[source]:
        del knowledge_base[source][destination]
        print(f"Fact Updated: Highway from '{source}' to '{destination}' blocked.")

# Simulating the agent learning new facts
add_new_route_fact("Nairobi", "Nakuru", 155.0) # E.g., a direct new expressway
block_route("Naivasha", "Nakuru") # Roadblock detected on standard route
display_kb()

Added Knowledge: Highway from 'Nairobi' to 'Nakuru' is 155.0 km.
Fact Updated: Highway from 'Naivasha' to 'Nakuru' blocked.

--- Current Environmental Knowledge Base (Map) ---
Highway: Nairobi      ➜ Naivasha     (Distance: 92.8 km)
Highway: Nairobi      ➜ Thika        (Distance: 43.0 km)
Highway: Nairobi      ➜ Narok        (Distance: 142.0 km)
Highway: Nairobi      ➜ Nakuru       (Distance: 155.0 km)
Highway: Thika        ➜ Nyeri        (Distance: 70.0 km)
Highway: Nyeri        ➜ Nakuru       (Distance: 102.0 km)
Highway: Narok        ➜ Bomet        (Distance: 98.0 km)
Highway: Bomet        ➜ Kisii        (Distance: 78.0 km)
Highway: Nakuru       ➜ Kericho      (Distance: 107.0 km)
Highway: Nakuru       ➜ Eldoret      (Distance: 157.0 km)
Highway: Kericho      ➜ Kisumu       (Distance: 83.9 km)
Highway: Eldoret      ➜ Kisumu       (Distance: 120.0 km)
Highway: Kisii        ➜ Kisumu       (Distance: 107.0 km)
--------------------------------------------------


In [3]:
def query_optimal_path(start, goal):
    """Inference engine using Uniform Cost Search (UCS) to query the KB."""
    if start not in knowledge_base or goal not in knowledge_base:
        return None, "Unknown location(s) provided."

    priority_queue = [(0, start, [start])]
    visited = {}
    
    while priority_queue:
        current_cost, current_node, path = heapq.heappop(priority_queue)
        
        if current_node == goal:
            return path, current_cost
            
        if current_node not in visited or current_cost < visited[current_node]:
            visited[current_node] = current_cost
            
            for neighbor, distance in knowledge_base.get(current_node, {}).items():
                new_cost = current_cost + distance
                heapq.heappush(priority_queue, (new_cost, neighbor, path + [neighbor]))

    return None, "No path found to the destination."

# Step 6: Current observation
current_location = "Nairobi"
destination_goal = "Kisumu"

# Step 7: Querying the agent
print("\n=== Self-Driving Trailer Routing System ===")
print(f"Agent is currently at: {current_location}")
print(f"Agent goal is: {destination_goal}")
path, cost = query_optimal_path(current_location, destination_goal)

if path:
    print(f"Agent Decision -> Optimal Path: {' ➜ '.join(path)} | Total Cost: {cost:.1f} km")
else:
    print(f"Agent Decision -> {cost}")


=== Self-Driving Trailer Routing System ===
Agent is currently at: Nairobi
Agent goal is: Kisumu
Agent Decision -> Optimal Path: Nairobi ➜ Nakuru ➜ Kericho ➜ Kisumu | Total Cost: 345.9 km


In [4]:
print("\n--- Additional Agent Queries ---")
queries = [
    ("Thika", "Kisumu"),
    ("Narok", "Kisumu"),
    ("Nakuru", "Nairobi"), # Testing a one-way directed graph failure
]

for start, end in queries:
    path, result = query_optimal_path(start, end)
    if path:
        print(f"Query ({start} ➜ {end}): Path found ({result:.1f} km): {' ➜ '.join(path)}")
    else:
        print(f"Query ({start} ➜ {end}): Route Failed - {result}")


--- Additional Agent Queries ---
Query (Thika ➜ Kisumu): Path found (362.9 km): Thika ➜ Nyeri ➜ Nakuru ➜ Kericho ➜ Kisumu
Query (Narok ➜ Kisumu): Path found (283.0 km): Narok ➜ Bomet ➜ Kisii ➜ Kisumu
Query (Nakuru ➜ Nairobi): Route Failed - No path found to the destination.


In [6]:
print("\n--- Debugging the Knowledge Base ---")
test_cases = [
    ("Nairobi", "Mombasa"),  # Unknown destination
    ("Kampala", "Kisumu"),   # Unknown origin
    ("Kisii", "Kisii")       # Same start and end
]

for start, end in test_cases:
    path, result = query_optimal_path(start, end)
    status = "SUCCESS" if path else "ERROR"
    print(f"[{status}] {start:10} ➜ {end:10} | Result: {result}")


--- Debugging the Knowledge Base ---
[ERROR] Nairobi    ➜ Mombasa    | Result: Unknown location(s) provided.
[ERROR] Kampala    ➜ Kisumu     | Result: Unknown location(s) provided.
[SUCCESS] Kisii      ➜ Kisii      | Result: 0
